# Database (Supabase PostgreSQL)


In [118]:
!pip install psycopg2-binary sqlalchemy python-dotenv -q

In [95]:
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

load_dotenv('.env')  # baca kredensial dari file .env, JANGAN hardcode di sini

DB_HOST = os.getenv('SUPABASE_DB_HOST')
DB_PORT = os.getenv('SUPABASE_DB_PORT')
DB_NAME = os.getenv('SUPABASE_DB_NAME')
DB_USER = os.getenv('SUPABASE_DB_USER')
DB_PASSWORD = os.getenv('SUPABASE_DB_PASSWORD')

DB_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DB_URL)

with engine.connect() as conn:
    print("✅ Koneksi berhasil!")
    print(conn.execute(text("SELECT version()")).scalar())


✅ Koneksi berhasil!
PostgreSQL 17.6 on aarch64-unknown-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit


In [130]:
with open('../sql/schema.sql') as f:
    schema_sql = f.read()

raw_conn = engine.raw_connection()
try:
    cur = raw_conn.cursor()
    cur.execute(schema_sql)
    raw_conn.commit()
    print('Schema berhasil dieksekusi penuh.')
finally:
    raw_conn.close()

with engine.connect() as conn:
    tables = conn.execute(text("SELECT table_name FROM information_schema.tables WHERE table_schema='public' AND table_type='BASE TABLE' ORDER BY table_name;")).fetchall()
print('Tabel:', [t[0] for t in tables])

Schema berhasil dieksekusi penuh.
Tabel: ['clean_comments', 'dashboard_summary', 'labeled_comments', 'model_results', 'prediction_history', 'preprocessed_comments', 'raw_comments']


In [131]:
df_raw = pd.read_parquet('../data/raw/raw_comments_kopdes.parquet')
cols = ['video_id','comment_id','parent_comment_id','level','username','nickname','comment','create_time']
df_raw[cols].to_sql('raw_comments', engine, if_exists='append', index=False, chunksize=5000, method='multi')
print(f'raw_comments: {len(df_raw):,} baris dimuat')

raw_comments: 220,051 baris dimuat


In [132]:
df_clean = pd.read_parquet('../data/interim/clean_comments_kopdes.parquet')
df_clean[cols].to_sql('clean_comments', engine, if_exists='append', index=False, chunksize=5000, method='multi')
print(f'clean_comments: {len(df_clean):,} baris dimuat')

clean_comments: 120,707 baris dimuat


In [133]:
df_prep = pd.read_parquet('../data/interim/preprocessed_comments_kopdes.parquet')
df_prep[['video_id','comment_id','text_final','tokens']].to_sql('preprocessed_comments', engine, if_exists='append', index=False, chunksize=5000, method='multi')
print(f'preprocessed_comments: {len(df_prep):,} baris dimuat')

preprocessed_comments: 118,778 baris dimuat


In [134]:
df_labeled = pd.read_parquet('../data/processed/labeled_comments_kopdes.parquet')
df_labeled[['video_id','comment_id','comment','text_for_bert','sentiment_label','sentiment_confidence','create_time']].to_sql(
    'labeled_comments', engine, if_exists='append', index=False, chunksize=5000, method='multi')
print(f'labeled_comments: {len(df_labeled):,} baris dimuat')

labeled_comments: 119,565 baris dimuat


In [135]:
df_mr = pd.read_csv('../data/processed/model_results.csv')
rename_map = {'Model':'model_name','Accuracy':'accuracy','Macro Precision':'macro_precision',
              'Macro Recall':'macro_recall','Macro F1':'macro_f1','Weighted Precision':'weighted_precision',
              'Weighted Recall':'weighted_recall','Weighted F1':'weighted_f1'}
df_mr = df_mr.rename(columns=rename_map)
keep = [c for c in ['model_name','dataset','accuracy','macro_precision','macro_recall','macro_f1',
                     'weighted_precision','weighted_recall','weighted_f1','n_test_samples','is_best_model'] if c in df_mr.columns]
df_mr[keep].to_sql('model_results', engine, if_exists='append', index=False)
print(f'model_results: {len(df_mr):,} baris dimuat')

model_results: 3 baris dimuat


In [136]:
raw_conn = engine.raw_connection()
try:
    cur = raw_conn.cursor()
    cur.execute('UPDATE model_results SET is_best_model = FALSE;')
    cur.execute("UPDATE model_results SET is_best_model = TRUE WHERE id = (SELECT id FROM model_results ORDER BY macro_f1 DESC LIMIT 1);")
    cur.execute("""
        INSERT INTO dashboard_summary (metric_scope, metric_name, metric_value)
        SELECT 'global', 'total_raw_comments', COUNT(*) FROM raw_comments
        UNION ALL SELECT 'global', 'total_clean_comments', COUNT(*) FROM clean_comments
        UNION ALL SELECT 'global', 'total_labeled_comments', COUNT(*) FROM labeled_comments
        UNION ALL SELECT 'global', 'total_unique_videos', COUNT(DISTINCT video_id) FROM raw_comments;
    """)
    cur.execute("""
        INSERT INTO dashboard_summary (metric_scope, video_id, metric_name, metric_value)
        SELECT 'per_video', video_id, 'positive_pct', positive_pct FROM v_sentiment_by_video
        UNION ALL SELECT 'per_video', video_id, 'total_comments', total_comments FROM v_sentiment_by_video;
    """)
    raw_conn.commit()
    print('UPDATE & dashboard_summary selesai.')
finally:
    raw_conn.close()

UPDATE & dashboard_summary selesai.


In [137]:
with engine.connect() as conn:
    for tbl in ['raw_comments','clean_comments','preprocessed_comments','labeled_comments','model_results','dashboard_summary']:
        n = conn.execute(text(f'SELECT COUNT(*) FROM {tbl};')).scalar()
        print(f'{tbl:25s}: {n:,} baris')

raw_comments             : 220,051 baris
clean_comments           : 120,707 baris
preprocessed_comments    : 118,778 baris
labeled_comments         : 119,565 baris
model_results            : 3 baris
dashboard_summary        : 58 baris


In [138]:
import shutil
import os
from google.colab import files

zip_source = 'sentiment-kopdes'
if os.path.exists(zip_source):
    shutil.rmtree(zip_source)
os.makedirs(zip_source, exist_ok=True)

# Folder-folder project yang mau disertakan (sesuai struktur GitHub)
folders_to_include = ['data', 'models', 'reports', 'scripts', 'sql']

print('=== Menyalin folder ===')
for folder in folders_to_include:
    if os.path.exists(folder):
        dest = os.path.join(zip_source, folder)
        shutil.copytree(folder, dest, dirs_exist_ok=True)
        n_files = sum(len(files_) for _, _, files_ in os.walk(dest))
        print(f'✅ {folder}/  ({n_files} file)')
    else:
        print(f'⚠️  Dilewati (belum ada): {folder}')

# Bikin zip
zip_filename = 'sentiment-kopdes-project.zip'
if os.path.exists(zip_filename):
    os.remove(zip_filename)
shutil.make_archive('sentiment-kopdes-project', 'zip', zip_source)

size_mb = os.path.getsize(zip_filename) / 1e6
print()
print(f'Ukuran zip: {size_mb:.2f} MB')
print()

# Rincian ukuran per folder (biar tahu mana yang berat)
print('=== Rincian ukuran per folder ===')
for folder in folders_to_include:
    path = os.path.join(zip_source, folder)
    if os.path.exists(path):
        total = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fn in os.walk(path) for f in fn)
        print(f'{folder:12s}: {total/1e6:.2f} MB')

if size_mb < 150:
    files.download(zip_filename)
    print()
    print('Download otomatis dimulai...')
else:
    print()
    print('⚠️  File > 150MB, kemungkinan gagal auto-download lewat browser.')
    print('Ambil manual: panel Files (sidebar kiri) -> klik kanan')
    print(f'"{zip_filename}" -> Download')
    print('Atau simpan ke Drive dulu:')
    print(f'  shutil.copy("{zip_filename}", "/content/drive/MyDrive/{zip_filename}")')

=== Menyalin folder ===
✅ data/  (17 file)
✅ models/  (6 file)
✅ reports/  (27 file)
✅ scripts/  (1 file)
✅ sql/  (1 file)

Ukuran zip: 100.18 MB

=== Rincian ukuran per folder ===
data        : 180.31 MB
models      : 1.59 MB
reports     : 2.19 MB
scripts     : 0.29 MB
sql         : 0.01 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download otomatis dimulai...
